In [1]:
import os
import requests
from llama_index.core import SimpleDirectoryReader
import fitz
import pymupdf4llm as ppdfLLm

In [ ]:
pdf_dir = "pdfs"
output_file = "output.txt"
pdf_url_list_file = "data.txt"

os.makedirs(pdf_dir, exist_ok=True)

with open(pdf_url_list_file, "r") as f:
    pdf_urls = [line.strip() for line in f if line.strip()]

for i, url in enumerate(pdf_urls):
    response = requests.get(url)
    if response.status_code == 200:
        file_path = os.path.join(pdf_dir, f"doc_{i+1}.pdf")
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f"Downloaded: {file_path}")
    else:
        print(f"Failed to download: {url}")


In [2]:
with open("data.txt", "r", encoding="utf-8") as f: 
    ds = f.readlines()


with open("pdfData.txt", "a", encoding="utf-8") as f: 
    for x in range(len(ds)): 
        dd = ds[x].split("/")[-1]
        pdfName = dd.split(".pdf")[0]
        
        ppdf_reader = ppdfLLm.LlamaMarkdownReader()
        z = x+1
        book = ppdf_reader.load_data("pdfs/doc_"+str(z)+".pdf")

        joinBookContent =  f"\n {pdfName}: || dataContent: ".join([jp.text.replace("\n", " ")for jp in book])
        bookLine = pdfName + " || dataContent: " + joinBookContent
        f.write(bookLine + "\n")

Successfully imported LlamaIndex
Processing pdfs/doc_1.pdf...
[                                        ] (0/1=======================================[========================================] (1/1]
Processing pdfs/doc_1.pdf...
[                                        ] (0/1=======================================[========================================] (1/1]
Processing pdfs/doc_1.pdf...
[                                        ] (0/1=======================================[========================================] (1/1]
Processing pdfs/doc_1.pdf...
[                                        ] (0/1=======================================[========================================] (1/1]
Processing pdfs/doc_1.pdf...
[                                        ] (0/1=======================================[========================================] (1/1]
Processing pdfs/doc_1.pdf...
[                                        ] (0/1=======================================[===============================

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


def text_formatter(text:str): 
    text = text.replace("\n", "")
    text = text.replace("  ", "")
    return text

In [8]:
def open_and_read_file():
    pages_and_chunks  = []
     
    with open("pdfData.txt", "r",  encoding="utf-8") as f:
        lines = f.readlines()
        for line in lines:
            strip_line = line.strip()
            
            metadatas, contexts = strip_line.split("|| dataContent:")
            
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=250,  
                chunk_overlap=20
            )
            documents = splitter.split_text(contexts) 
            for doc in documents:
                
                formattedText = text_formatter(doc)
                if len(formattedText)<=10: 
                    continue
                chunk_dict = {}
                chunk_dict["line_data"] = "metadata: " + metadatas + " ||content:" + formattedText.replace("#", "")
                chunk_dict["line_char_count"] = len(formattedText)
                chunk_dict["line_word_count"] = len(doc.strip().split())
      
                pages_and_chunks.append(chunk_dict)
    return pages_and_chunks

In [9]:
arrData = open_and_read_file()
print(arrData)

[{'line_data': 'metadata: cfad-annual-faculty-exhibition  ||content:**ANNUAL FACULTY SHOW** College of Fine Arts & Design, University of Sharjah YEARNING**Etihad Modern Art Gallery** **Abu Dhabi, April – May 2024** -----', 'line_char_count': 153, 'line_word_count': 25}, {'line_data': 'metadata: cfad-annual-faculty-exhibition:  ||content: YEARNING -----', 'line_char_count': 16, 'line_word_count': 3}, {'line_data': 'metadata: cfad-annual-faculty-exhibition:  ||content:Curator **Muatasim Alkubaisy**Editor **Nadia M. Alhasani**Art Director **Nada Abdallah**Catalog Designer **Zaid S. Dimiati**Photographer **Renji Mathews**Supporters **Mazin Alsamman** **Abdulhadi Alsalti** **Joey Abiertas** **Noushad', 'line_char_count': 233, 'line_word_count': 27}, {'line_data': 'metadata: cfad-annual-faculty-exhibition:  ||content:**Noushad Kadavath**Copyright © 2024 **College of Fine Arts and Design** **University of Sharjah**Printed in Sharjah United Arab Emirates -----', 'line_char_count': 143, 'line_w